In [ ]:
"""
TB-SERS Analyzer Preprocessing Pipeline

Steps:
1. Cosmic ray removal (Modified Z-score)
2. PCA-based outlier detection (Mahalanobis distance)
3. Baseline correction (Polynomial)
4. Shift minimum to zero
5. SNV normalization (per spectrum)
6. Averaging

Input: Raman spectra (.txt)
Output: CSV + QC plots
"""

import numpy as np
import matplotlib.pyplot as plt
from tkinter import filedialog
import os
import csv

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.covariance import MinCovDet
from scipy import stats
from pybaselines import Baseline

# ================= CONFIG =================
COSMIC_THRESHOLD = 8
PCA_COMPONENTS = 5
OUTLIER_CONFIDENCE = 0.90
POLY_ORDER = 3
# =========================================


# -----------------------------
# Cosmic Ray Removal
# -----------------------------
def modified_z_score(ys):
    median_y = np.median(ys)
    mad = np.median(np.abs(ys - median_y))
    
    if mad == 0:
        return np.zeros_like(ys)
    return 0.6745 * (ys - median_y) / mad


def fixer(y):
    mz = modified_z_score(y)
    spikes = np.abs(mz) > COSMIC_THRESHOLD
    y_out = y.copy()

    for i in range(len(spikes)):
        if spikes[i]:
            if 0 < i < len(y) - 1:
                y_out[i] = np.median([y[i - 1], y[i + 1]])
            elif i == 0:
                y_out[i] = y[i + 1]
            else:
                y_out[i] = y[i - 1]
    return y_out


# -----------------------------
# Detect spectrum length
# -----------------------------
def detect_num_points(data):
    wavenumbers = data[:, 2]
    first_val = wavenumbers[0]

    for i in range(1, len(wavenumbers)):
        if wavenumbers[i] == first_val:
            return i

    return len(wavenumbers)


# -----------------------------
# Main Pipeline
# -----------------------------
def main():

    # -----------------------------
    # Select Files
    # -----------------------------
    filePaths = filedialog.askopenfilename(
        multiple=True, title="Select Raman Data Files"
    )

    if not filePaths:
        print("❌ No files selected")
        return

    # -----------------------------
    # Select Output Folder
    # -----------------------------
    save_dir = filedialog.askdirectory(title="Select Output Folder")

    if not save_dir:
        print("❌ No output folder selected")
        return

    output_csv = os.path.join(save_dir, "sers_processed.csv")
    plot_dir = os.path.join(save_dir, "plots")

    os.makedirs(plot_dir, exist_ok=True)

    file_exists = os.path.exists(output_csv)

    with open(output_csv, "a+", newline="", encoding="utf-8") as f:

        writer = csv.writer(f)
        header_written = file_exists

        for idx, filePath in enumerate(filePaths):

            print(f"\n[{idx+1}/{len(filePaths)}] Processing: {filePath}")

            base_filename = os.path.splitext(os.path.basename(filePath))[0]

            # -----------------------------
            # Load Data
            # -----------------------------
            try:
                data = np.loadtxt(filePath)
            except Exception as e:
                print(f"❌ Error loading file: {filePath}")
                continue

            rows, cols = data.shape

            # -----------------------------
            # detect spectrum length
            # -----------------------------
            num_points = detect_num_points(data)

            if rows % num_points != 0:
                print("⚠ rows mismatch -> skip")
                continue

            num_spectra = rows // num_points

            print("points:", num_points)
            print("spectra:", num_spectra)

            wavenumber = data[:num_points, 2][::-1]
            intensities = data[:, 3].reshape(num_spectra, num_points)[:, ::-1]

            # -----------------------------
            # STEP 1 Cosmic removal
            # -----------------------------
            despiked = np.array([fixer(spec) for spec in intensities])

            # -----------------------------
            # STEP 2 PCA Outlier detection
            # -----------------------------
            scaler = StandardScaler()
            scaled = scaler.fit_transform(despiked)

            pca = PCA(n_components=PCA_COMPONENTS)
            scores = pca.fit_transform(scaled)

            robust_cov = MinCovDet().fit(scores)
            mahal = robust_cov.mahalanobis(scores)

            cutoff = stats.chi2.ppf(OUTLIER_CONFIDENCE, df=PCA_COMPONENTS)
            mask = mahal < cutoff

            filtered = despiked[mask]

            print("after outlier removal:", len(filtered))

            if len(filtered) == 0:
                print("⚠ all spectra removed")
                continue

            # -----------------------------
            # STEP 3 + 4 Baseline + Shift
            # -----------------------------
            processed = []

            for spec in filtered:
                baseline = Baseline().poly(spec, poly_order=POLY_ORDER)[0]
                corrected = spec - baseline
                corrected = corrected - np.min(corrected)
                processed.append(corrected)

            processed = np.array(processed)

            # -----------------------------
            # STEP 5 SNV normalization (FIXED)
            # -----------------------------
            mean_val = np.mean(processed, axis=1, keepdims=True)
            std_val = np.std(processed, axis=1, keepdims=True)

            std_val[std_val == 0] = 1  # กันหาร 0

            processed = (processed - mean_val) / std_val

            # -----------------------------
            # STEP 6 Averaging
            # -----------------------------
            final_average = np.mean(processed, axis=0)

            # -----------------------------
            # Write header
            # -----------------------------
            if not header_written:
                header = list(wavenumber) + ["Groups", "Filename"]
                writer.writerow(header)
                header_written = True

            row = np.concatenate((final_average, [""], [base_filename]))
            writer.writerow(row)

            # -----------------------------
            # Plot QC
            # -----------------------------
            plt.figure(figsize=(21, 6))

            plt.subplot(1, 3, 1)
            plt.plot(wavenumber, intensities.T, alpha=0.2)
            plt.title("Original spectra")

            plt.subplot(1, 3, 2)
            plt.plot(wavenumber, processed.T, alpha=0.1)
            plt.title("Processed spectra")

            plt.subplot(1, 3, 3)
            plt.plot(wavenumber, final_average, linewidth=2)
            plt.title("Final average")

            plt.tight_layout()

            plt.savefig(
                os.path.join(plot_dir, f"{base_filename}.png"),
                dpi=300
            )

            plt.close()

    print("\n✅ Processing Complete")


# -----------------------------
# Run
# -----------------------------
if __name__ == "__main__":
    main()